# Hot Stamping Intelligence: Tensile Strength Prediction via ElasticNet Regression

---
**Project 11 · LozanoLsa Industrial ML Portfolio**  
**Domain:** Automotive Manufacturing — Hot Stamping / Press Hardening  
**Algorithm:** ElasticNet Regression (L1 + L2 combined · ElasticNetCV for joint alpha + l1_ratio selection)  
**Target:** `tensile_strength_mpa` — Part tensile strength after press hardening (MPa, continuous)

---

## 🏭 Business Context

Hot stamping — press hardening a high-strength steel blank at temperatures above 900°C and quenching it inside the die — is the dominant manufacturing process for automotive structural safety parts: B-pillars, door reinforcements, bumper beams. The mechanical properties of the finished part, particularly tensile strength (MPa), are determined by a complex interaction of thermal, mechanical, and material variables.

The challenge facing the process engineer is not a lack of sensors — modern hot stamping lines log a dozen or more parameters at every stroke. The challenge is that some variables are **correlated** (furnace entry and exit temperatures track each other at r = 0.95), some are **genuinely irrelevant** and should be discarded, and some carry real independent signal. The right regularisation method depends on the actual data structure — and **that is not known in advance**.

**ElasticNet is the right framework for this problem precisely because it does not require you to bet on L1 or L2 upfront.** By combining both penalties and selecting the optimal mixture via 5-fold cross-validation (ElasticNetCV), it lets the data decide. In this case, the CV grid confirms that the process is Lasso-like: `l1_ratio = 1.0` achieves the best CV score, meaning the dataset rewards sparsity over coefficient shrinkage. One feature is zeroed entirely (`furnace_entry_temp_c`), captured redundantly by `furnace_exit_temp_c`.

This project delivers:
1. A high-accuracy tensile strength predictor (R² = 0.893 on held-out test set)
2. A principled comparison between ElasticNet, Ridge, and Lasso on this specific process
3. A recipe simulator quantifying the impact of steel grade selection, die cooling, and press force on the final part strength

---

*"You don't need to know which regularisation to use — you need a framework that finds out for you."*


## 1 · Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import stats
from scipy.stats import normaltest, probplot

from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet, ElasticNetCV, Ridge, Lasso, LinearRegression
from sklearn.linear_model import enet_path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_goldfeldquandt

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

C_BLUE  = "#4C9BE8"
C_RED   = "#E8574C"
C_GREEN = "#2ECC71"
C_AMBER = "#F39C12"
C_DARK  = "#1B2840"
C_MUTED = "#697888"
C_PURP  = "#9B59B6"

FEATURES = [
    "furnace_entry_temp_c", "furnace_exit_temp_c", "die_temp_c",
    "transfer_time_s", "press_force_kn", "press_speed_mm_s",
    "dwell_time_s", "blank_thickness_mm", "steel_grade",
    "active_cooling", "cooling_pressure_bar", "cooling_flow_l_min",
]
TARGET = "tensile_strength_mpa"

# Steel grade reference
STEEL_GRADES = {1: "Usibor (1500 MPa class)", 2: "Ductibor (500 MPa class)", 3: "MBorian (700 MPa class)"}
SPEC_MIN = 900   # typical minimum spec for structural B-pillar (MPa)

print(f"✓ Environment ready — {len(FEATURES)} process variables")
print(f"  Minimum tensile spec: {SPEC_MIN} MPa")


## 2 · Load Data

The dataset contains around 1,500 hot stamping cycles captured from the press PLC, furnace SCADA, and inline tensile testing records. Each row represents one production stroke — from blank loading through die quench to part ejection.

| Column | Type | Description |
|---|---|---|
| `furnace_entry_temp_c` | float | Blank temperature entering the furnace (°C) |
| `furnace_exit_temp_c` | float | Blank temperature exiting the furnace (°C) — correlated with entry |
| `die_temp_c` | float | Active die surface temperature (°C) |
| `transfer_time_s` | float | Time between furnace exit and die close (s) |
| `press_force_kn` | float | Maximum press force during forming (kN) |
| `press_speed_mm_s` | float | Ram closing speed (mm/s) |
| `dwell_time_s` | float | Time blank remains in closed die (s) |
| `blank_thickness_mm` | float | Sheet blank thickness (mm) |
| `steel_grade` | int | 1 = Usibor · 2 = Ductibor · 3 = MBorian |
| `active_cooling` | int | 1 = water-cooled die active · 0 = passive |
| `cooling_pressure_bar` | float | Cooling water pressure (bar) |
| `cooling_flow_l_min` | float | Cooling water flow rate (L/min) |
| `tensile_strength_mpa` | float | **Target** — part tensile strength after quench (MPa) |


In [ ]:
try:
    df = pd.read_csv("hot_stamping_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/ElasticNet_Hot_Stamping/main/hot_stamping_data.csv")

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nSteel grade distribution:")
for g, label in {1:'Usibor',2:'Ductibor',3:'MBorian'}.items():
    n = (df['steel_grade']==g).sum()
    print(f"  Grade {g} ({label:16}): {n:,} records ({n/len(df)*100:.1f}%)")
print(f"\nActive cooling: {df['active_cooling'].mean()*100:.1f}% of cycles")
df.head()


## 3 · Sanity Checks

In [ ]:
print("── Shape ────────────────────────────")
print(f"  Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")

print("\n── Data Types ──────────────────────")
print(df.dtypes.to_string())

print("\n── Descriptive Statistics ──────────")
display(df.describe().round(2).T)


In [ ]:
print("── Missing / Duplicates ────────────")
print(f"  Missing: {df.isna().sum().sum()}" if df.isna().any().any() else "  ✓ No missing values.")
print(f"  Duplicates: {df.duplicated().sum()}" if df.duplicated().any() else "  ✓ No duplicates.")

print("\n── Target Summary ──────────────────")
print(f"  tensile_strength_mpa: [{df[TARGET].min():.1f}, {df[TARGET].max():.1f}] MPa")
print(f"  Mean ± Std: {df[TARGET].mean():.1f} ± {df[TARGET].std():.1f} MPa")
print(f"  Within spec (≥ {SPEC_MIN} MPa): {(df[TARGET]>=SPEC_MIN).mean()*100:.1f}%")
print(f"\n── Strength by Steel Grade ─────────")
for g, label in STEEL_GRADES.items():
    sub = df[df['steel_grade']==g][TARGET]
    print(f"  Grade {g} ({label}): {sub.mean():.1f} ± {sub.std():.1f} MPa")

print(f"\n── Key Collinearity Signal ─────────")
r_temps = df['furnace_entry_temp_c'].corr(df['furnace_exit_temp_c'])
print(f"  furnace_entry_temp_c ↔ furnace_exit_temp_c: r = {r_temps:.3f}  ← ElasticNet motivation")


## 4 · Exploratory Data Analysis

Two stories dominate this EDA:

1. **Press force is the dominant mechanical driver** — with r = 0.88 with the target, it dwarfs every other variable. This is metallurgically consistent: forming pressure during the quench determines martensite transformation completeness.

2. **The furnace temperature collinearity problem** — entry and exit temperatures correlate at r = 0.95. This is not a data artefact; it reflects real process physics. The furnace exit temperature adds information only at the margin beyond what entry temperature already captured. ElasticNet will decide whether to keep both, zero one, or shrink both.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── (A) Strength distribution by steel grade
ax = axes[0]
palette_g = {1: C_BLUE, 2: C_GREEN, 3: C_AMBER}
for g, label in STEEL_GRADES.items():
    vals = df[df['steel_grade']==g][TARGET]
    ax.hist(vals, bins=30, alpha=0.65, color=palette_g[g],
            label=f"Gr.{g} {label[:7]}", edgecolor="white", lw=0.3)
ax.axvline(SPEC_MIN, color=C_RED, ls="--", lw=1.8, label=f"Spec ≥{SPEC_MIN} MPa")
ax.set_xlabel("Tensile Strength (MPa)", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("Strength Distribution by Steel Grade", fontsize=11, fontweight="bold")
ax.legend(fontsize=7)

# ── (B) Press force vs strength (dominant driver)
ax2 = axes[1]
for g, color in palette_g.items():
    mask = df['steel_grade']==g
    ax2.scatter(df[mask]['press_force_kn'], df[mask][TARGET],
                alpha=0.25, s=8, color=color, label=f"Gr.{g}")
m, b, r, *_ = stats.linregress(df['press_force_kn'], df[TARGET])
xr = np.linspace(df['press_force_kn'].min(), df['press_force_kn'].max(), 100)
ax2.plot(xr, m*xr+b, color=C_RED, lw=1.5, ls="--", label=f"r = {r:.3f}")
ax2.axhline(SPEC_MIN, color=C_RED, lw=1.0, ls=":", alpha=0.6)
ax2.set_xlabel("Press Force (kN)", fontsize=10)
ax2.set_ylabel("Tensile Strength (MPa)", fontsize=10)
ax2.set_title("Press Force vs Strength\n(dominant driver, r = 0.88)", fontsize=11, fontweight="bold")
ax2.legend(fontsize=7)

# ── (C) The collinearity problem: furnace temps
ax3 = axes[2]
ax3.scatter(df['furnace_entry_temp_c'], df['furnace_exit_temp_c'],
            c=df[TARGET], cmap="RdYlGn", alpha=0.30, s=8)
m2, b2, r2, *_ = stats.linregress(df['furnace_entry_temp_c'], df['furnace_exit_temp_c'])
xr2 = np.linspace(df['furnace_entry_temp_c'].min(), df['furnace_entry_temp_c'].max(), 100)
ax3.plot(xr2, m2*xr2+b2, color="white", lw=1.5, ls="--", label=f"r = {r2:.3f}")
ax3.set_xlabel("Furnace Entry Temp (°C)", fontsize=10)
ax3.set_ylabel("Furnace Exit Temp (°C)", fontsize=10)
ax3.set_title(f"Collinearity: Entry ↔ Exit Temp\nr = {r2:.3f} — ElasticNet motivation", fontsize=11, fontweight="bold")
ax3.legend(fontsize=8)

plt.suptitle("Hot Stamping — Target Overview & Key Process Relationships",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Correlation ranking (univariate) ─────────────────────────────────
corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bar_c = [C_RED if v > 0 else C_BLUE for v in corr.values]
bars = ax.barh(corr.index[::-1], corr.values[::-1], color=bar_c[::-1],
               alpha=0.80, edgecolor="none", height=0.65)
ax.axvline(0, color=C_DARK, lw=0.8)
for bar, val in zip(bars, corr.values[::-1]):
    offset = 0.01 if val >= 0 else -0.01
    ax.text(val+offset, bar.get_y()+bar.get_height()/2,
            f"{val:+.3f}", va="center",
            ha="left" if val>=0 else "right", fontsize=9, color=C_DARK)
ax.set_xlabel("Pearson Correlation with tensile_strength_mpa", fontsize=11)
ax.set_title("Univariate Feature Correlation Ranking\n"
             "(ElasticNet will refine this accounting for inter-variable effects)",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("Note: steel_grade has NEGATIVE correlation — higher grade number = lower strength class")
print("  Grade 1 (Usibor) = highest strength  |  Grade 3 (MBorian) = lowest")


In [ ]:
# ── VIF — quantify the collinearity problem ──────────────────────────
X_vif = sm.add_constant(df[FEATURES])
vif_df = pd.DataFrame({
    "Feature": FEATURES,
    "VIF": [variance_inflation_factor(X_vif.values, i+1) for i in range(len(FEATURES))]
}).sort_values("VIF", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bar_c = [C_RED if v > 10 else C_AMBER if v > 5 else C_BLUE for v in vif_df["VIF"]]
bars = ax.barh(vif_df["Feature"][::-1], vif_df["VIF"][::-1],
               color=bar_c[::-1], alpha=0.82, edgecolor="none", height=0.60)
ax.axvline(10, color=C_RED, lw=1.2, ls="--", label="VIF = 10 (severe)")
ax.axvline(5,  color=C_AMBER, lw=1.0, ls=":", label="VIF = 5 (moderate)")
for bar, val in zip(bars, vif_df["VIF"][::-1]):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, f"{val:.1f}",
            va="center", fontsize=9)
ax.set_xlabel("Variance Inflation Factor (VIF)", fontsize=11)
ax.set_title("Collinearity Diagnosis — VIF per Feature\n"
             "furnace_entry/exit temps require regularisation", fontsize=11, fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

display(vif_df.style.format({"VIF": "{:.2f}"}))
print("\nHigh VIF features are candidates for regularisation.")
print("ElasticNet decides whether to shrink or zero them — the L1/L2 ratio controls this tradeoff.")


## 5 · Preprocessing

ElasticNet **requires standardisation** — both the L1 and L2 penalties are scale-sensitive. Without scaling, `press_force_kn` (values ∼1200) would receive a proportionally smaller penalty than `dwell_time_s` (values ∼3.5), distorting the regularisation result. `StandardScaler` brings all features to the same scale before the penalty is applied.


In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Scaling: StandardScaler (mandatory for ElasticNet)")
print(f"\nRaw scale comparison (why scaling matters):")
print(f"  press_force_kn  std = {X_train['press_force_kn'].std():.1f} kN")
print(f"  dwell_time_s    std = {X_train['dwell_time_s'].std():.2f} s")
print(f"  After scaling: both σ = 1.0  ← penalty applied equitably")


## 6 · Model Training

### The ElasticNet penalty landscape

ElasticNet combines the L1 (Lasso) and L2 (Ridge) penalties:

$$\text{Loss} = \sum_{i=1}^{n}(y_i - \hat{y}_i)^2 + \alpha \left[ \rho \sum_{j}|\beta_j| + \frac{1-\rho}{2} \sum_{j}\beta_j^2 \right]$$

Two hyperparameters must be selected:
- **α** — overall regularisation strength
- **ρ (l1_ratio)** — the L1/L2 mix: 0 = pure Ridge, 1 = pure Lasso, 0.5 = balanced

**ElasticNetCV** searches a joint grid of α values × l1_ratio values via 5-fold cross-validation, selecting the combination that maximises CV R². This is the principled, reproducible approach — no manual tuning.

**What the CV found:** the optimal l1_ratio = 1.0 (pure L1/Lasso penalty at the selected α). This tells us the hot stamping data rewards sparsity — some features are genuinely redundant and should be zeroed, not just shrunk. The correlated furnace entry temperature is the primary candidate.


In [ ]:
# ── ElasticNetCV — joint search over alpha × l1_ratio ────────────────
l1_ratios_grid = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]
alphas_grid    = np.logspace(-3, 2, 60)

en_cv = ElasticNetCV(
    l1_ratio=l1_ratios_grid,
    alphas=alphas_grid,
    cv=5,
    random_state=42,
    max_iter=20_000
)
en_cv.fit(X_train_sc, y_train)

best_alpha  = en_cv.alpha_
best_l1     = en_cv.l1_ratio_

print(f"ElasticNetCV — 5-fold CV results:")
print(f"  Evaluated : {len(alphas_grid)} alpha × {len(l1_ratios_grid)} l1_ratio candidates")
print(f"  Best alpha    : {best_alpha:.5f}")
print(f"  Best l1_ratio : {best_l1:.2f}  ({'pure Lasso (L1)' if best_l1==1.0 else 'mixed L1+L2' if best_l1>0 else 'pure Ridge (L2)'})")
print(f"\n  Interpretation:")
print(f"    l1_ratio = 1.0 → the data rewards sparsity over coefficient shrinkage.")
print(f"    The furnace temperature redundancy is better resolved by elimination (L1)")
print(f"    than by proportional shrinkage (L2).")


In [ ]:
# ── ElasticNet path across l1_ratios ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette_l1 = [C_BLUE, C_GREEN, C_AMBER, C_RED, C_PURP, "#1ABC9C", "#E67E22", "white"]

# Left: path for l1_ratio = best (1.0) — show coefficient trajectories
alphas_path = np.logspace(-1, 2, 80)
coefs_path  = []
for a in alphas_path:
    m = ElasticNet(alpha=a, l1_ratio=best_l1, max_iter=20_000)
    m.fit(X_train_sc, y_train)
    coefs_path.append(m.coef_)
coefs_path = np.array(coefs_path)

ax = axes[0]
for i, feat in enumerate(FEATURES):
    ax.plot(np.log10(alphas_path), coefs_path[:, i], lw=1.2, alpha=0.85, label=feat)
ax.axvline(np.log10(best_alpha), color="white", lw=2.0, ls="--",
           label=f"Optimal α = {best_alpha:.4f}")
ax.axhline(0, color=C_DARK, lw=0.5, ls=":")
ax.set_xlabel("log₁₀(α)  →  Stronger regularisation", fontsize=10)
ax.set_ylabel("Standardised Coefficient (MPa per σ)", fontsize=10)
ax.set_title(f"ElasticNet Path (l1_ratio = {best_l1:.2f})\nSome features zeroed at optimal α",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=6.5, ncol=2, loc="lower right")

# Right: R² vs l1_ratio at optimal alpha
ax2 = axes[1]
r2_l1 = []
for l1 in l1_ratios_grid:
    m = ElasticNet(alpha=best_alpha, l1_ratio=l1, max_iter=20_000)
    m.fit(X_train_sc, y_train)
    r2_l1.append(r2_score(y_test, m.predict(X_test_sc)))

ax2.plot(l1_ratios_grid, r2_l1, color=C_BLUE, marker="o", lw=2, markersize=7)
ax2.axvline(best_l1, color=C_RED, lw=1.8, ls="--",
            label=f"Optimal l1_ratio = {best_l1:.2f}")
ax2.set_xlabel("l1_ratio  (0 = pure Ridge ← → pure Lasso = 1)", fontsize=10)
ax2.set_ylabel("R² Test Set", fontsize=10)
ax2.set_title("R² vs l1_ratio at Optimal Alpha\n"
              "CV confirms Lasso behaviour is best for this data", fontsize=11, fontweight="bold")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ── Fit final ElasticNet ─────────────────────────────────────────────
en = ElasticNet(alpha=best_alpha, l1_ratio=best_l1, max_iter=20_000, random_state=42)
en.fit(X_train_sc, y_train)
y_train_pred = en.predict(X_train_sc)
y_test_pred  = en.predict(X_test_sc)

print(f"✓ ElasticNet fitted: alpha={best_alpha:.5f}  l1_ratio={best_l1:.2f}")

# Feature selection result
coef_df = pd.DataFrame({
    "Feature"    : FEATURES,
    "Coefficient": en.coef_,
    "Abs_Impact" : np.abs(en.coef_)
}).sort_values("Abs_Impact", ascending=False).reset_index(drop=True)

active = coef_df[coef_df["Coefficient"] != 0]
zeroed = coef_df[coef_df["Coefficient"] == 0]

print(f"\nFeature selection result:")
print(f"  Active: {len(active)}/{len(FEATURES)}  |  Zeroed: {len(zeroed)}/{len(FEATURES)}")
print(f"  Zeroed: {zeroed['Feature'].tolist()}")
print(f"  → furnace_entry_temp_c eliminated — redundant with furnace_exit_temp_c (r=0.95)")


## 7 · Model Evaluation

| Metric | ElasticNet | Ridge | Lasso | Operational Meaning |
|---|---|---|---|---|
| **R²** | **0.893** | 0.891 | 0.893 | 89.3% of tensile variance explained — all three regularisations perform similarly |
| **RMSE** | **43.24 MPa** | 44.1 MPa | 43.3 MPa | Typical prediction error relative to 900 MPa spec — 4.4% relative error on 900 MPa spec |
| **MAE** | **34.64 MPa** | 35.2 MPa | 34.7 MPa | Median miss — adequate for process recipe guidance |

The three regularisation methods converge to virtually identical accuracy. This is expected: when the true signal is strong (R² ≈ 0.89), regularisation penalises mostly noise — all three methods find the same dominant relationships. The value of ElasticNet is not higher accuracy; it is that **the framework selected the correct penalty automatically** without requiring the engineer to know in advance whether the data was Ridge-like or Lasso-like.


In [ ]:
r2_train = r2_score(y_train, y_train_pred)
r2_test  = r2_score(y_test, y_test_pred)
rmse     = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae      = mean_absolute_error(y_test, y_test_pred)
residuals= y_test.values - y_test_pred

print("ElasticNet — Performance Summary")
print(f"  R² train:   {r2_train:.5f}")
print(f"  R² test:    {r2_test:.5f}")
print(f"  RMSE test:  {rmse:.2f} MPa")
print(f"  MAE  test:  {mae:.2f} MPa")
print(f"  RMSE / Mean strength: {rmse/y.mean()*100:.2f}% relative error")

# Full comparison table
print("\n── Regularisation Comparison (same scaler, same split) ────────────")
rows = []
for name, m in [
    ("ElasticNet (α=CV, ρ=CV)", en),
    ("Ridge (α=1.0)",  Ridge(alpha=1.0)),
    ("Lasso (α=0.01)", Lasso(alpha=0.01, max_iter=20_000)),
    ("OLS (no reg.)",  LinearRegression()),
]:
    if name != "ElasticNet (α=CV, ρ=CV)":
        m.fit(X_train_sc, y_train)
    yp = m.predict(X_test_sc)
    rows.append({"Model": name,
                 "R² Test": round(r2_score(y_test, yp), 5),
                 "RMSE (MPa)": round(np.sqrt(mean_squared_error(y_test, yp)), 3),
                 "MAE (MPa)": round(mean_absolute_error(y_test, yp), 3)})

display(pd.DataFrame(rows).style.highlight_max(subset=["R² Test"], color="#d5f5e3")
                              .highlight_min(subset=["RMSE (MPa)", "MAE (MPa)"], color="#d5f5e3"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── (A) Predicted vs Actual
ax = axes[0]
in_spec = y_test >= SPEC_MIN
ax.scatter(y_test[in_spec],  y_test_pred[in_spec],
           alpha=0.45, s=12, color=C_BLUE, label="Within spec")
ax.scatter(y_test[~in_spec], y_test_pred[~in_spec],
           alpha=0.45, s=12, color=C_RED, label="Below spec")
lims = [y_test.min()-10, y_test.max()+10]
ax.plot(lims, lims, color=C_DARK, ls="--", lw=1.5, label="Perfect")
ax.axvline(SPEC_MIN, color=C_AMBER, lw=1.2, ls=":", alpha=0.8)
ax.axhline(SPEC_MIN, color=C_AMBER, lw=1.2, ls=":", alpha=0.8, label=f"Spec ≥{SPEC_MIN} MPa")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("Actual (MPa)", fontsize=11)
ax.set_ylabel("Predicted (MPa)", fontsize=11)
ax.set_title(f"Predicted vs Actual — R² = {r2_test:.4f}", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)

# ── (B) Residuals vs Fitted
ax2 = axes[1]
ax2.scatter(y_test_pred, residuals, alpha=0.35, s=10,
            c=[C_RED if abs(r) > 60 else C_BLUE for r in residuals])
ax2.axhline(0, color=C_DARK, lw=1.5, ls="--")
ax2.axhline(+2*rmse, color=C_AMBER, lw=1, ls=":", label=f"±2·RMSE = ±{2*rmse:.1f} MPa")
ax2.axhline(-2*rmse, color=C_AMBER, lw=1, ls=":")
ax2.set_xlabel("Fitted (MPa)", fontsize=11)
ax2.set_ylabel("Residual (MPa)", fontsize=11)
ax2.set_title("Residuals vs Fitted", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 8 · Interpretability

ElasticNet standardised coefficients measure the change in tensile strength (MPa) for a one-standard-deviation increase in each feature. The coefficient hierarchy maps directly to press hardening metallurgy:

- **Press force dominates** — the pressure during die quench drives martensite transformation completeness. Insufficient force allows retained austenite, which weakens the part.
- **Steel grade is decisive at recipe level** — Usibor (grade 1) is engineered for 1500 MPa class; MBorian (grade 3) is a 700 MPa class. No process adjustment can compensate for the wrong steel selection.
- **Die temperature (negative)** — hotter dies slow quench rate, reducing martensite fraction. Die temperature management is the primary tool for dialling in strength when steel grade is fixed.
- **furnace_entry_temp_c is zeroed** — its signal is fully captured by furnace_exit_temp_c (r = 0.95). Monitoring one thermocouple is sufficient; the second is redundant.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

cs = coef_df.sort_values("Coefficient", ascending=True)
bar_c = [C_MUTED if c == 0 else (C_RED if c > 0 else C_BLUE) for c in cs["Coefficient"]]
bars = ax.barh(cs["Feature"], cs["Coefficient"], color=bar_c, alpha=0.82, edgecolor="none", height=0.65)


for bar, val, feat in zip(bars, cs["Coefficient"], cs["Feature"]):
    if val == 0:
        bar.set_hatch("////"); bar.set_alpha(0.30)
        ax.text(0.5, bar.get_y()+bar.get_height()/2,
                "zeroed →", va="center", ha="left", fontsize=8, color=C_MUTED, style="italic")
    else:
        offset = 1.0 if val >= 0 else -1.0
        ax.text(val+offset, bar.get_y()+bar.get_height()/2,
                f"{val:+.2f}", va="center",
                ha="left" if val >= 0 else "right", fontsize=9, color=C_DARK)

ax.axvline(0, color=C_DARK, lw=0.8)
ax.set_xlabel("Standardised ElasticNet Coefficient (MPa per σ input change)", fontsize=11)
ax.set_title(f"ElasticNet Importance — {len(active)}/{len(FEATURES)} Active · {len(zeroed)} Zeroed (α={best_alpha:.4f}, ρ={best_l1:.2f}, 5-fold CV)", fontsize=11, fontweight="bold")



ax.text(0.99, 0.01, "Red = increases strength  |  Blue = decreases strength  |  Hatched = zeroed",
        transform=ax.transAxes, fontsize=8, color=C_MUTED, ha="right")
plt.tight_layout()
plt.show()

print("Active features (sorted by impact):")
for _, row in coef_df[coef_df["Coefficient"]!=0].iterrows():
    d = "▲" if row["Coefficient"]>0 else "▼"
    print(f"  {row['Feature']:<25}: {row['Coefficient']:+.4f} MPa/σ  {d}")
print(f"Zeroed: {zeroed['Feature'].tolist()}")



In [ ]:
pi = permutation_importance(en, X_test_sc, y_test, n_repeats=30, random_state=42, scoring="r2")

pi_df = pd.DataFrame({"Feature":FEATURES,
                       "Importance":pi.importances_mean,
                       "Std":pi.importances_std})          .sort_values("Importance", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 5))
bar_c2 = [C_RED if v>0.1 else C_AMBER if v>0.01 else C_BLUE for v in pi_df["Importance"]]
ax.barh(pi_df["Feature"][::-1], pi_df["Importance"][::-1],
        xerr=pi_df["Std"][::-1], color=bar_c2[::-1],
        alpha=0.82, edgecolor="none", height=0.60, capsize=4)
ax.axvline(0, color=C_DARK, lw=0.8)
ax.set_xlabel("Mean R² drop when feature permuted (higher = more important)", fontsize=10)
ax.set_title("Permutation Feature Importance — ElasticNet Model (30 reps · ±1 std)", fontsize=11, fontweight="bold")


plt.tight_layout(); plt.show()

display(pi_df.style.format({"Importance":"{:.5f}","Std":"{:.5f}"}))


## 9 · Statistical Validation

ElasticNet is a biased estimator — the combined L1+L2 penalty intentionally biases coefficients toward zero to reduce variance. Classical p-value inference (as available from OLS via statsmodels) does not directly apply. Residual diagnostics, however, remain valid and test whether the model has captured the process structure adequately.


In [ ]:
from scipy.stats import normaltest

dag_stat, dag_p = normaltest(residuals)
print(f"D'Agostino-Pearson normality (n={len(residuals)}):")
print(f"  stat={dag_stat:.4f}  p={dag_p:.4f}")
print(f"  → {'Residuals normally distributed.' if dag_p>0.05 else 'Mild non-normality — acceptable given RMSE=24 MPa.'}")

dw = durbin_watson(residuals)
print(f" Durbin-Watson: {dw:.4f}  (ideal ≈ 2)")

print(f"  → {'No autocorrelation.' if 1.5<dw<2.5 else 'Check record ordering.'}")

X_te_c = sm.add_constant(X_test_sc)
gq_stat, gq_p, _ = het_goldfeldquandt(residuals, X_te_c)
print(f"Goldfeld-Quandt homoscedasticity: stat={gq_stat:.4f}  p={gq_p:.4f}")

print(f"  → {'Constant variance.' if gq_p>0.05 else 'Some heteroscedasticity — consider log-transform of target.'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sns.histplot(residuals, bins=25, kde=True, color=C_BLUE, alpha=0.7, ax=ax)
ax.axvline(0, color=C_RED, ls="--", lw=1.5)
ax.set_xlabel("Residual (MPa)"); ax.set_title("Residual Distribution", fontsize=12, fontweight="bold")

ax2 = axes[1]
(osm, osr), (slope, intercept, r) = probplot(residuals, dist="norm")
ax2.scatter(osm, osr, s=10, color=C_BLUE, alpha=0.65)
xl = np.linspace(min(osm), max(osm), 100)
ax2.plot(xl, slope*xl+intercept, color=C_RED, lw=1.5, ls="--")
ax2.set_xlabel("Theoretical Quantiles"); ax2.set_ylabel("Sample Quantiles")
ax2.set_title("Normal Q-Q Plot", fontsize=12, fontweight="bold")

plt.tight_layout(); plt.show()


## 10 · Process Simulator

Three scenarios demonstrate the levers at the metallurgist's disposal. The dominant message is that **steel grade selection is a first-principle decision** that no process adjustment can override — but within a given grade, die cooling and press force are the primary controllable knobs.

| Scenario | Story | Expected |
|---|---|---|
| **A — Usibor, optimised recipe** | Grade 1 + high force + active cooling | Strong (>1200 MPa) |
| **B — MBorian, unfavourable conditions** | Grade 3 + passive cooling + under-force | Weak (<900 MPa) |
| **C — MBorian, corrected cooling** | Grade 3 + active cooling + adjusted temps | Recovered |


In [ ]:
def predict_strength(**kwargs):
    """Predict tensile strength for a given process configuration.
    Unspecified features default to dataset medians."""
    base = df[FEATURES].median().to_dict()
    base.update(kwargs)
    x = pd.DataFrame([[base[c] for c in FEATURES]], columns=FEATURES)
    strength = en.predict(scaler.transform(x))[0]
    status   = "✅ Within spec" if strength >= SPEC_MIN else "⚠ Below spec"
    margin   = strength - SPEC_MIN
    return strength, status, margin


In [ ]:
# ── Scenario A — Usibor, optimised ───────────────────────────────────
a_mpa, a_st, a_mg = predict_strength( steel_grade=1, furnace_exit_temp_c=920, press_force_kn=1350,

    die_temp_c=140, transfer_time_s=4.0, active_cooling=1,
    cooling_pressure_bar=17, cooling_flow_l_min=36, blank_thickness_mm=1.0
)
print("═══ Scenario A — Usibor (Gr.1) · High Force · Active Cooling ════")
print(f"  Steel: Usibor 1500 MPa class | Force: 1350 kN | Die: 140°C | Cooling: ON")
print(f"  → Tensile Strength: {a_mpa:.1f} MPa   {a_st}")
print(f"  → Margin above spec: {a_mg:+.1f} MPa")

# ── Scenario B — MBorian, unfavourable ───────────────────────────────
b_mpa, b_st, b_mg = predict_strength(
    steel_grade=3, furnace_exit_temp_c=870, press_force_kn=1050,
    die_temp_c=165, transfer_time_s=6.5, active_cooling=0,
    blank_thickness_mm=1.6
)
print("═══ Scenario B — MBorian (Gr.3) · Low Force · Passive Cooling ═══")

print(f"  Steel: MBorian 700 MPa class | Force: 1050 kN | Die: 165°C | Cooling: OFF")
print(f"  → Tensile Strength: {b_mpa:.1f} MPa   {b_st}")
print(f"  → Margin above spec: {b_mg:+.1f} MPa")
print(f"  ⚠ Passive cooling + under-force: insufficient quench rate.")

# ── Scenario C — MBorian, corrected cooling ──────────────────────────
c_mpa, c_st, c_mg = predict_strength(
    steel_grade=3, furnace_exit_temp_c=870, press_force_kn=1050,
    die_temp_c=145, transfer_time_s=4.5, active_cooling=1,
    cooling_pressure_bar=16, cooling_flow_l_min=33, blank_thickness_mm=1.6
)
print("═══ Scenario C — MBorian (Gr.3) · Corrected Recipe ══════════════")

print(f"  Steel: MBorian (same grade) | Force: 1050 kN | Die: 145°C | Cooling: ON")
print(f"  → Tensile Strength: {c_mpa:.1f} MPa   {c_st}")
print(f"  → Margin above spec: {c_mg:+.1f} MPa")
print("  ✅ Activating cooling and reducing die temp (+20°C margin)")

print(f"     recovered {c_mpa-b_mpa:.1f} MPa — from {b_mpa:.1f} to {c_mpa:.1f} MPa.")
print(f"     Note: MBorian targets lower strength class — within its own spec range.")


In [ ]:
# ── Response Surface: Press Force × Die Temperature ──────────────────
force_range = np.linspace(900, 1700, 60)
die_temp_range = np.linspace(130, 180, 60)
F, D = np.meshgrid(force_range, die_temp_range)

grid = pd.DataFrame({ "furnace_entry_temp_c": df["furnace_entry_temp_c"].median(),

    "furnace_exit_temp_c" : 900.0,
    "die_temp_c"          : D.ravel(),
    "transfer_time_s"     : 5.0,
    "press_force_kn"      : F.ravel(),
    "press_speed_mm_s"    : 45.0,
    "dwell_time_s"        : 3.5,
    "blank_thickness_mm"  : 1.2,
    "steel_grade"         : 1,    # Usibor
    "active_cooling"      : 1,
    "cooling_pressure_bar": 14.0,
    "cooling_flow_l_min"  : 30.0,
})
Z = en.predict(scaler.transform(grid[FEATURES])).reshape(F.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(F, D, Z, levels=25, cmap="RdYlGn", alpha=0.88)
cbar = plt.colorbar(cf, ax=ax); cbar.set_label("Predicted Tensile Strength (MPa)")
cs = ax.contour(F, D, Z, levels=[SPEC_MIN], colors=["white"], linewidths=2.0)
ax.clabel(cs, fmt=f"{SPEC_MIN} MPa spec", fontsize=9, colors="white")
ax.contourf(F, D, Z, levels=[SPEC_MIN, 2000], colors=["lime"], alpha=0.15, hatches=["////"])
ax.set_xlabel("Press Force (kN)", fontsize=11)
ax.set_ylabel("Die Temperature (°C)", fontsize=11)
ax.set_title("Tensile Strength Response Surface — Usibor (Gr.1) · Active Cooling\nGreen hatch = combinations meeting spec", fontsize=11, fontweight="bold")



plt.tight_layout(); plt.show()

print("Practical insight: higher press force or lower die temperature")
print("both push strength up — operators can compensate one with the other.")


## 11 · Final Reflection

---

### What ElasticNet found

Twelve process variables enter the model; eleven carry independent predictive signal. The exception — `furnace_entry_temp_c` — is zeroed because its information is fully captured by `furnace_exit_temp_c` (r = 0.95). The L1 component of ElasticNet identified this redundancy and removed it cleanly, just as a process engineer would: if you already know the exit temperature, monitoring the entry thermocouple adds nothing to your prediction of tensile strength.

The coefficient hierarchy is metallurgically grounded:

- **Press force (+114 MPa/σ)** — the single strongest driver. Forming pressure determines how completely the austenite-to-martensite transformation occurs during the die quench. Under-pressure = retained austenite = weakened part.
- **Steel grade (−50 MPa/σ)** — the grade encoding means grade 3 (MBorian) produces lower absolute strength than grade 1 (Usibor). This is by design: MBorian is a ductility-optimised grade. The model correctly recovers the metallurgical hierarchy.
- **Die temperature (−28 MPa/σ)** — a hotter die slows the quench, reducing martensite fraction. Die temperature management is the operator's primary adjustment tool for tensile outcome.

### The ElasticNet lesson

The CV grid searched 60 × 8 = 480 hyperparameter combinations and returned l1_ratio = 1.0 — pure Lasso behaviour. This is not a failure of ElasticNet; it is the correct answer for this dataset. The value of the ElasticNet framework is that it would also have found l1_ratio = 0.5 or 0.0 if the data warranted it. The engineer does not need to decide — the data decides.

For comparison: a naive Ridge model at alpha = 1.0 achieves the same R² but keeps `furnace_entry_temp_c` active with a small non-zero coefficient. That coefficient would mislead a process engineer into monitoring a redundant sensor. ElasticNet zeroed it and told the truth.

### Limits

The model's accuracy (RMSE = 24 MPa) is meaningful relative to the 900 MPa spec level — a 4.4% relative error on 900 MPa spec. However, it does not account for blank-to-blank material variability within a coil (carbon equivalent, grain size), tooling wear effects on heat transfer, or part geometry complexity (draw depth, corner radii affect local cooling rates). For safety-critical B-pillar applications, the model prediction should gate into a decision support workflow, not into an autonomous process controller.

---

*LozanoLsa | Operational Excellence · MBB · Machine Learning | GitHub: LozanoLsa*
